#### Data Load

In [1]:
import os
import sys
repo_root = os.path.abspath(os.path.join(os.getcwd(),"../../.."))
sys.path.insert(0,repo_root)

In [2]:
from Gen_AI.rag_systems.services.text_extractor import TextExtractor
from Gen_AI.config import pdf_path

In [3]:
file_path = str(pdf_path)

In [4]:
document_extract = TextExtractor()
text_data  =document_extract.convert_to_markdown(data=file_path,page_metadata=False,verbose=True)

---Entered markdown---
---Entered detect_file_type---
------The Detected Extension of the file is: pdf------
---Extracting Text from PDF---


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

KeyboardInterrupt: 

In [ ]:
text_data

#### Chunking and Evaluation

In [ ]:
import tiktoken
import math
from langchain_text_splitters import RecursiveCharacterTextSplitter
def text_splitter(text: str, embedding_model_token_limit:int,encoding_model:str='gpt-3.5-turbo',chunk_overlap_percent=0.05,verbose=False):
    try:
        encoding = tiktoken.encoding_for_model(encoding_model)
        total_tokens = len(encoding.encode(text))
        print(f"Total Tokens: {total_tokens}")
        
        if total_tokens>embedding_model_token_limit:
            Max_chunks = math.ceil(total_tokens/embedding_model_token_limit)
            Total_char=len(text.replace('\n', ''))
            Max_Char_len_per_chunk = math.ceil(Total_char/Max_chunks)
        else:
            Max_chunks = 1
            Total_char=len(text.replace('\n', ''))
            Max_Char_len_per_chunk = math.ceil(Total_char/Max_chunks)
        if verbose:
            print(f"-------Started text splitting-------\nTotal_Tokens:{total_tokens}\n\nEstimated_Chunk_split:{Max_chunks}\n\nMaximum Characters per chunk:{Max_Char_len_per_chunk}\n\n-------------")
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=Max_Char_len_per_chunk,
            chunk_overlap=(Max_Char_len_per_chunk*chunk_overlap_percent),
            separators=["\n\n", "\n"],
            )
        if verbose:
            print(f"-------Text splitting Complete-------")
        if total_tokens>embedding_model_token_limit:
            return text_splitter.split_text(text)
        else:
            return [text]
    
    except Exception as e:
        if verbose:
            print.error(f"Error: {e}")
            print(f"Error: {e}")
        return None

In [ ]:
chunks = text_splitter(text=text_data,embedding_model_token_limit=1800,chunk_overlap_percent=0.10,verbose=True)

In [ ]:
print(chunks)

In [ ]:
from statistics import mean, median, stdev

def evaluate_chunk_size(chunks: list[str]):
    chunk_length = [len(chunk) for chunk in chunks]

    return {
        "chunk_count":len(chunks),
        "mean_Chunk_char":round(mean(chunk_length),4),
        "median_chunk_char":round(median(chunk_length),4),
        "min_chunk_char":round(min(chunk_length),4),
        "max_chunk_char": round(max(chunk_length),4),
        "std_chunk_char": round(stdev(chunk_length),4)

        if len(chunk_length)>1 else 0

    }

In [ ]:
evaluation = evaluate_chunk_size(chunks=chunks)

In [ ]:
print(evaluation)

In [ ]:
import numpy as np
def chunk_percentile(chunks: list[str]):
    chunk_length = [len(chunk) for chunk in chunks]

    return {
        "25_percentile": np.percentile(chunk_length,25),
        "50_percentile":np.percentile(chunk_length,50),
        "75_percentile":np.percentile(chunk_length,75),
        "90_percentile":np.percentile(chunk_length,90),
        "95_percentile":np.percentile(chunk_length,95),
        "99_percentile":np.percentile(chunk_length,99),
    }

In [ ]:
evaluate_percentile = chunk_percentile(chunks=chunks)
print(evaluate_percentile)

In [ ]:
def outlier_ratio(chunks):

    lengths = [len(chunk) for chunk in chunks]

    threshold = 2 * np.median(lengths)

    outliers = [
        x
        for x in lengths
        if x > threshold
    ]

    return {
        "outlier_count": len(outliers),
        "outlier_ratio": len(outliers)/len(lengths)
    }

In [ ]:
outlier = outlier_ratio(chunks)
print(outlier)

In [ ]:
sorted(
    [len(chunk) for chunk in chunks],
    reverse=True
)[:10]

In [ ]:
import re

def sentence_boundary_violation_rate(chunks):

    sentence_endings = r'[.!?]["\']?\s*$'

    violating_chunks = []

    for idx, chunk in enumerate(chunks):

        if not re.search(sentence_endings, chunk.strip()):
            violating_chunks.append(idx)

    return {
        "violation_rate": len(violating_chunks) / len(chunks),
        "violating_chunks": violating_chunks
    }

In [ ]:
rate = sentence_boundary_violation_rate(chunks)

print(rate)

In [ ]:
for i in range(5):
    print(f"\nChunk {i}")
    print(repr(chunks[i][-100:]))

In [ ]:
def non_linguistic_content_ratio(chunk):

    alpha_chars = sum(
        c.isalpha()
        for c in chunk
    )

    return 1 - (alpha_chars / len(chunk))

In [ ]:
test = non_linguistic_content_ratio(chunks)
print(test)

In [ ]:
import re

def meaningful_word_ratio(chunk):

    words = re.findall(r'\b[a-zA-Z]{2,}\b', chunk)

    total_tokens = len(chunk.split())

    if total_tokens == 0:
        return 0

    return len(words) / total_tokens

In [ ]:
for chunk in chunks:
    test = meaningful_word_ratio(chunk)
    print('===========================')
    print(test)

In [ ]:
from statistics import mean, median


def evaluate_meaningful_word_ratio(chunks):

    scores = [
        meaningful_word_ratio(chunk)
        for chunk in chunks
    ]

    return {
        "mean_meaningful_word_ratio": round(mean(scores), 4),
        "median_meaningful_word_ratio": round(median(scores), 4),
        "min_meaningful_word_ratio": round(min(scores), 4),
        "max_meaningful_word_ratio": round(max(scores), 4),
    }

In [ ]:
test = evaluate_meaningful_word_ratio(chunks)

print(test)

In [ ]:
def find_suspicious_chunks(chunks, threshold=1.0):

    suspicious = []

    for idx, chunk in enumerate(chunks):

        score = meaningful_word_ratio(chunk)

        if score > threshold:
            suspicious.append(
                {
                    "chunk_index": idx,
                    "score": score,
                    "preview": chunk[:150]
                }
            )

    return suspicious

In [ ]:
test = find_suspicious_chunks(chunks)
print(test)

In [ ]:
import re

def is_base64_chunk(chunk):

    return bool(
        re.search(
            r'iVBORw0KGgo|/9j/4AAQSkZJRg',
            chunk
        )
    )

In [ ]:
def base64_chunk_ratio(chunks):

    count = sum(
        is_base64_chunk(chunk)
        for chunk in chunks
    )

    return count / len(chunks)

In [ ]:
test = base64_chunk_ratio(chunks)
print(test)

In [ ]:
import re

def is_table_chunk(chunk):

    table_patterns = [
        r'\|.*\|',          # markdown table row
        r'\|[-:\s]+\|',     # markdown separator row
    ]

    matches = 0

    for pattern in table_patterns:
        if re.search(pattern, chunk):
            matches += 1

    return matches > 0

In [ ]:
def is_table_chunk(chunk, min_rows=6):
    table_rows = [
        line
        for line in chunk.splitlines()
        if "|" in line
    ]

    return len(table_rows) >= min_rows

In [ ]:
def table_chunk_ratio(chunks):

    table_chunks = []

    for idx, chunk in enumerate(chunks):

        if is_table_chunk(chunk):
            table_chunks.append(idx)

    return {
        "table_chunk_count": len(table_chunks),
        "table_chunk_ratio": len(table_chunks) / len(chunks),
        "table_chunk_indices": table_chunks
    }

In [ ]:
test = table_chunk_ratio(chunks)
print(test)

In [ ]:
def table_density(chunk):

    lines = chunk.splitlines()

    table_lines = sum(
        "|" in line
        for line in lines
    )

    return table_lines / max(1, len(lines))

In [ ]:
sorted(
    [
        table_density(chunk)
        for chunk in chunks
    ],
    reverse=True
)[:10]

In [ ]:
import re

def is_code_chunk(chunk):

    patterns = [
        r'```',                 # markdown code fence
        r'def\s+\w+\(',         # python function
        r'class\s+\w+',         # class definition
        r'import\s+\w+',        # import statement
        r'from\s+\w+\s+import',
        r'if\s+__name__',
        r'public\s+class',      # Java
        r'function\s+\w+\(',    # JS
        r'console\.log',
        r'SELECT\s+.*FROM',     # SQL
    ]

    for pattern in patterns:

        if re.search(
            pattern,
            chunk,
            re.IGNORECASE
        ):
            return True

    return False    

In [ ]:
def code_chunk_ratio(chunks):

    code_chunks = []

    for idx, chunk in enumerate(chunks):

        if is_code_chunk(chunk):
            code_chunks.append(idx)

    return {
        "code_chunk_count": len(code_chunks),
        "code_chunk_ratio": len(code_chunks) / len(chunks),
        "code_chunk_indices": code_chunks
    }

In [ ]:
test  = code_chunk_ratio(chunks)
print(test)

In [ ]:
from collections import Counter

def duplicate_chunk_ratio(chunks):

    counts = Counter(chunks)

    duplicates = sum(
        count - 1
        for count in counts.values()
        if count > 1
    )

    return duplicates / len(chunks)

In [ ]:
test  = duplicate_chunk_ratio(chunks)
print(test)

In [ ]:
import re

def lexical_coverage_ratio(
    original_text,
    chunks
):

    original_words = set(
        re.findall(r'\w+', original_text.lower())
    )

    chunk_words = set(
        re.findall(
            r'\w+',
            " ".join(chunks).lower()
        )
    )

    return (
        len(chunk_words & original_words)
        /
        len(original_words)
    )

In [ ]:
test = lexical_coverage_ratio(text_data,chunks)
print(test)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

#### Extraction Evaluators

In [5]:
text_data  =document_extract.convert_to_markdown(data=file_path,page_metadata=True,verbose=True)

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


---Entered markdown---
---Entered detect_file_type---
------The Detected Extension of the file is: pdf------
---Extracting Text from PDF---


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

---Text Extraction Complete---
------Exiting Markdown------


In [6]:
print(text_data)

[{'content': "Imagine a security guard at a building entrance. He: Observes the environment (sees who approaches, checks ID) Decides based on rules or judgment (allow or deny entry) Acts (opens the gate, calls for backup, logs an entry) This loop — perceive → decide → act — never stops. That is exactly what an agent is. An agent is not just a program. It is an autonomous entity embedded in an environment,\ncontinuously sensing it and affecting it through actions. An agent is defined by the tuple: Where: = Perception function: \n — maps environment state to an internal percept = Decision function (policy): \n — maps perceived state to an action = Action space — the set of all possible actions = Environment — what the agent operates in The agent function maps the entire percept history to an action: This is important — an agent's decision can depend on everything it has ever seen, not just the\ncurrent moment. MODULE 1 — Foundations of Agents 🧠 Concept 1: The Agent Definition Intuition F

In [7]:
print(len(text_data))

167


In [8]:
for page in text_data:
    print(page['content'])

Imagine a security guard at a building entrance. He: Observes the environment (sees who approaches, checks ID) Decides based on rules or judgment (allow or deny entry) Acts (opens the gate, calls for backup, logs an entry) This loop — perceive → decide → act — never stops. That is exactly what an agent is. An agent is not just a program. It is an autonomous entity embedded in an environment,
continuously sensing it and affecting it through actions. An agent is defined by the tuple: Where: = Perception function: 
 — maps environment state to an internal percept = Decision function (policy): 
 — maps perceived state to an action = Action space — the set of all possible actions = Environment — what the agent operates in The agent function maps the entire percept history to an action: This is important — an agent's decision can depend on everything it has ever seen, not just the
current moment. MODULE 1 — Foundations of Agents 🧠 Concept 1: The Agent Definition Intuition Formal Explanation


In [9]:
def empty_page_ratio(pages):

    empty_pages = []

    for page in pages:

        if len(page["content"].strip()) == 0:

            empty_pages.append(
                page["page_num"]
            )

    return {
        "empty_page_count":
            len(empty_pages),

        "empty_page_ratio":
            len(empty_pages) /
            len(pages),

        "empty_pages":
            empty_pages
    }

In [10]:
test = empty_page_ratio(text_data)
print(test)

{'empty_page_count': 0, 'empty_page_ratio': 0.0, 'empty_pages': []}


In [11]:
from statistics import mean, median, stdev


def character_density(pages):

    page_lengths = [
        len(page["content"])
        for page in pages
    ]

    return {
        "mean_chars_per_page":
            round(mean(page_lengths), 2),

        "median_chars_per_page":
            round(median(page_lengths), 2),

        "min_chars_per_page":
            min(page_lengths),

        "max_chars_per_page":
            max(page_lengths),

        "std_chars_per_page":
            round(
                stdev(page_lengths),
                2
            ) if len(page_lengths) > 1 else 0
    }

In [12]:
test = character_density(text_data)
print(test)

{'mean_chars_per_page': 1734.13, 'median_chars_per_page': 1311, 'min_chars_per_page': 174, 'max_chars_per_page': 90072, 'std_chars_per_page': 6899.94}


In [13]:
def suspicious_pages(
    pages,
    min_chars=100
):

    suspicious = []

    for idx,page in enumerate(pages):

        if len(page["content"]) < min_chars:

            suspicious.append({
                "page_num":
                    idx,

                "char_count":
                    len(page["content"])
            })

    return suspicious

In [14]:
test = suspicious_pages(text_data,min_chars=1000)
print(test)

[{'page_num': 0, 'char_count': 999}, {'page_num': 3, 'char_count': 261}, {'page_num': 5, 'char_count': 266}, {'page_num': 7, 'char_count': 471}, {'page_num': 11, 'char_count': 327}, {'page_num': 12, 'char_count': 334}, {'page_num': 17, 'char_count': 236}, {'page_num': 19, 'char_count': 260}, {'page_num': 21, 'char_count': 335}, {'page_num': 23, 'char_count': 334}, {'page_num': 24, 'char_count': 993}, {'page_num': 27, 'char_count': 337}, {'page_num': 30, 'char_count': 327}, {'page_num': 33, 'char_count': 281}, {'page_num': 35, 'char_count': 299}, {'page_num': 40, 'char_count': 303}, {'page_num': 42, 'char_count': 977}, {'page_num': 43, 'char_count': 177}, {'page_num': 45, 'char_count': 587}, {'page_num': 48, 'char_count': 413}, {'page_num': 51, 'char_count': 446}, {'page_num': 58, 'char_count': 442}, {'page_num': 62, 'char_count': 523}, {'page_num': 67, 'char_count': 400}, {'page_num': 71, 'char_count': 470}, {'page_num': 75, 'char_count': 463}, {'page_num': 78, 'char_count': 242}, {'pa

In [19]:
lengths = sorted(
    [
        (
            indx,
            len(page["content"])
        )
        for indx,page in enumerate(text_data)
    ],
    key=lambda x: x[1],
    reverse=False
)

print(lengths[:-10:-1])

[(1, 90072), (91, 2576), (54, 2226), (53, 2153), (41, 2134), (76, 2133), (56, 2111), (120, 2017), (123, 1977)]


In [23]:
import re


def contains_base64_artifact(text):

    known_signatures = [
        r'iVBORw0KGgo',     # PNG
        r'/9j/4AAQSkZJRg',  # JPEG
        r'R0lGOD',          # GIF
        r'UklGR'            # WEBP
    ]

    for sig in known_signatures:

        if re.search(sig, text):
            return True

    long_base64 = (
        r'[A-Za-z0-9+/]{200,}={0,2}'
    )

    return bool(
        re.search(long_base64, text)
    )

In [24]:
def base64_page_ratio(pages):

    flagged_pages = []

    for indx,page in enumerate(pages):

        if contains_base64_artifact(page["content"]):

            flagged_pages.append(
                indx
            )

    return {
        "base64_page_count":
            len(flagged_pages),

        "base64_page_ratio":
            len(flagged_pages) /
            len(pages),

        "base64_pages":
            flagged_pages
    }

In [25]:
test = base64_page_ratio(text_data)
print(test)

{'base64_page_count': 1, 'base64_page_ratio': 0.005988023952095809, 'base64_pages': [1]}


In [27]:
def detect_base64_pages(pages):

    findings = []

    for indx,page in enumerate(pages):

        if contains_base64_artifact(
            page["content"]
        ):

            findings.append(
                {
                    "page_num":
                        indx,

                    "char_count":
                        len(page["content"]),

                    "preview":
                        page["content"][:150]
                }
            )

    return {
        "base64_page_count":
            len(findings),

        "base64_page_ratio":
            len(findings) /
            len(pages),

        "pages":
            findings
    }

In [28]:
test = detect_base64_pages(text_data)
test

{'base64_page_count': 1,
 'base64_page_ratio': 0.005988023952095809,
 'pages': [{'page_num': 1,
   'char_count': 90072,
   'preview': 'iVBORw0KGgoAAAANSUhEUgAADGgAAAMgCAIAAABKhyKtAAEAAElEQVR4nOz93ZIdxbkvemdWjRbgab/BRmK/WvtoXIL3skQ0AiJ0S3zYp2vo1NNIt6RYBpa2EfvlEvpwRUyEO4ywLfWozDeyqruRQI'}]}

In [29]:
import re


def ocr_noise_ratio(text: str):

    total_chars = max(1, len(text))

    replacement_chars = len(
        re.findall(r'�', text)
    )

    unusual_symbols = len(
        re.findall(
            r'[□■▪▫◆◇◊¤]',
            text
        )
    )

    repeated_garbage = len(
        re.findall(
            r'[@#$%^&*]{4,}',
            text
        )
    )

    noise_count = (
        replacement_chars +
        unusual_symbols +
        repeated_garbage
    )

    return {
        "noise_ratio":
            round(
                noise_count /
                total_chars,
                4
            ),

        "noise_count":
            noise_count,

        "replacement_chars":
            replacement_chars,

        "unusual_symbols":
            unusual_symbols,

        "garbage_sequences":
            repeated_garbage
    }

In [43]:
def page_level_ocr_noise(pages):

    page_scores = []

    for indx,page in enumerate(pages):

        score = ocr_noise_ratio(
            page["content"]
        )
        if score["noise_ratio"] > 0:
            page_scores.append(
                {
                    "page_num":
                        indx,

                    **score
                }
            )
    return page_scores if len(page_scores)>0 else "No Noise Found"

In [44]:
test = page_level_ocr_noise(text_data)
print(test)

No Noise Found


In [45]:
import re

def meaningful_word_ratio(text):

    words = re.findall(
        r'\b[a-zA-Z]+\b',
        text
    )

    if not words:
        return {
            "meaningful_word_ratio": 0.0,
            "word_count": 0
        }

    meaningful = [
        w
        for w in words
        if len(w) >= 3
    ]

    ratio = len(meaningful) / len(words)

    return {
        "meaningful_word_ratio":
            round(ratio, 4),

        "meaningful_word_count":
            len(meaningful),

        "word_count":
            len(words)
    }

In [51]:
def page_level_meaningful_word_ratio(
    pages,
    threshold=0.5
):

    findings = []

    for indx,page in enumerate(pages):

        result = meaningful_word_ratio(
            page["content"]
        )

        if (
            result["meaningful_word_ratio"]
            > threshold
        ):

            findings.append(
                {
                    "page_num":
                        indx,

                    **result
                }
            )

    return findings

In [52]:
test = page_level_meaningful_word_ratio(text_data)
print(test)

[{'page_num': 0, 'meaningful_word_ratio': 0.7516, 'meaningful_word_count': 118, 'word_count': 157}, {'page_num': 1, 'meaningful_word_ratio': 0.6041, 'meaningful_word_count': 293, 'word_count': 485}, {'page_num': 2, 'meaningful_word_ratio': 0.8947, 'meaningful_word_count': 102, 'word_count': 114}, {'page_num': 3, 'meaningful_word_ratio': 0.8462, 'meaningful_word_count': 33, 'word_count': 39}, {'page_num': 4, 'meaningful_word_ratio': 0.8346, 'meaningful_word_count': 212, 'word_count': 254}, {'page_num': 5, 'meaningful_word_ratio': 0.8205, 'meaningful_word_count': 32, 'word_count': 39}, {'page_num': 6, 'meaningful_word_ratio': 0.813, 'meaningful_word_count': 187, 'word_count': 230}, {'page_num': 7, 'meaningful_word_ratio': 0.9077, 'meaningful_word_count': 59, 'word_count': 65}, {'page_num': 8, 'meaningful_word_ratio': 0.8312, 'meaningful_word_count': 197, 'word_count': 237}, {'page_num': 9, 'meaningful_word_ratio': 0.8222, 'meaningful_word_count': 148, 'word_count': 180}, {'page_num': 10,

In [53]:
def table_density(text):

    lines = text.splitlines()

    if not lines:
        return {
            "table_density": 0.0,
            "table_line_count": 0,
            "total_line_count": 0
        }

    table_lines = sum(
        "|" in line
        for line in lines
    )

    return {
        "table_density":
            round(
                table_lines / len(lines),
                4
            ),

        "table_line_count":
            table_lines,

        "total_line_count":
            len(lines)
    }

In [62]:
def page_level_table_density(
    pages,
    threshold=0.001
):

    findings = []

    for indx,page in enumerate(pages):

        result = table_density(
            page["content"]
        )

        if result["table_density"] > threshold:

            findings.append(
                {
                    "page_num":
                        indx,

                    **result
                }
            )

    return findings

In [63]:
test = page_level_table_density(text_data)
print(test)

[{'page_num': 3, 'table_density': 1.0, 'table_line_count': 7, 'total_line_count': 7}, {'page_num': 5, 'table_density': 1.0, 'table_line_count': 7, 'total_line_count': 7}, {'page_num': 7, 'table_density': 1.0, 'table_line_count': 11, 'total_line_count': 11}, {'page_num': 11, 'table_density': 1.0, 'table_line_count': 5, 'total_line_count': 5}, {'page_num': 12, 'table_density': 1.0, 'table_line_count': 6, 'total_line_count': 6}, {'page_num': 17, 'table_density': 1.0, 'table_line_count': 4, 'total_line_count': 4}, {'page_num': 19, 'table_density': 1.0, 'table_line_count': 6, 'total_line_count': 6}, {'page_num': 21, 'table_density': 1.0, 'table_line_count': 8, 'total_line_count': 8}, {'page_num': 23, 'table_density': 1.0, 'table_line_count': 7, 'total_line_count': 7}, {'page_num': 25, 'table_density': 0.6667, 'table_line_count': 14, 'total_line_count': 21}, {'page_num': 27, 'table_density': 1.0, 'table_line_count': 9, 'total_line_count': 9}, {'page_num': 30, 'table_density': 1.0, 'table_lin

In [64]:
import re


def detect_headers(text):

    patterns = [

        r'^#{1,6}\s+.+$',            # markdown

        r'^[A-Z][A-Z\s]{4,}$',       # ALL CAPS

        r'^\d+(\.\d+)*\s+.+$',       # 1 Intro / 1.1 Intro

        r'^Chapter\s+\d+.*$'         # Chapter 1
    ]

    headers = []

    for line in text.splitlines():

        line = line.strip()

        if not line:
            continue

        for pattern in patterns:

            if re.match(pattern, line):

                headers.append(line)

                break

    return headers

In [66]:
def header_metrics(text):

    headers = detect_headers(text)

    return {

        "header_count":
            len(headers),

        "unique_headers":
            len(set(headers)),

        "headers":
            headers[:20]
    }

In [68]:
def page_level_header(
    pages,
    threshold=0.001
):

    findings = []

    for indx,page in enumerate(pages):

        result = header_metrics(
            page["content"]
        )

        if result["header_count"] > 1:

            findings.append(
                {
                    "page_num":
                        indx,

                    **result
                }
            )

    return findings if len(findings)>0 else "No Header Found"

In [69]:
test = page_level_header(text_data)
print(test)

[{'page_num': 55, 'header_count': 3, 'unique_headers': 3, 'headers': ['500          1000', '500    │  (500, 500)  │  (500, 1000) │', '1000   │ (1000, 500)  │   (0, 0)    │◄── overflow']}, {'page_num': 68, 'header_count': 4, 'unique_headers': 4, 'headers': ['4 agents, 1 traitor  → POSSIBLE   (n=4 ≥ 3(1)+1)', '7 agents, 2 traitors → POSSIBLE   (n=7 ≥ 3(2)+1) In LLM multi-agent systems, a hallucinating agent is functionally a Byzantine general', '3 nodes → tolerates 1 failure', '5 nodes → tolerates 2 failures The Byzantine Generals Problem Practical Consensus: Raft Protocol']}, {'page_num': 112, 'header_count': 2, 'unique_headers': 2, 'headers': ['# Non-atomic (dangerous):', '# read-modify-write: 3 steps # Atomic (safe):']}, {'page_num': 113, 'header_count': 3, 'unique_headers': 3, 'headers': ['# Untyped — any agent can write anything', '# Overwritten by every agent — last write wins', '# Ambiguous — what values are valid? # GOOD STATE DESIGN']}, {'page_num': 114, 'header_count': 3, 'uniq

In [81]:
import re

def is_symbolic_line(line):

    alpha_count = sum(
        c.isalpha()
        for c in line
    )

    return alpha_count < 3

In [82]:
from collections import Counter

def repeated_line_ratio(text):

    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
        and not is_symbolic_line(line)
    ]

    counts = Counter(lines)

    repeated = sum(
        count
        for count in counts.values()
        if count > 1
    )

    return {"Repeate_Count":repeated / max(1, len(lines)),
    
    "line": lines
    }

In [83]:
def page_level_repeated_words(
    pages,
    threshold=0.001
):

    findings = []

    for indx,page in enumerate(pages):

        result = repeated_line_ratio(
            page["content"]
        )

        findings.append(
            {
                "page_num":
                    indx,

                **result
            }
        )

    return findings if len(findings)>0 else "No Repetition Found"

In [84]:
test = page_level_repeated_words(text_data)
print(test)

[{'page_num': 0, 'Repeate_Count': 0.0, 'line': ['Imagine a security guard at a building entrance. He: Observes the environment (sees who approaches, checks ID) Decides based on rules or judgment (allow or deny entry) Acts (opens the gate, calls for backup, logs an entry) This loop — perceive → decide → act — never stops. That is exactly what an agent is. An agent is not just a program. It is an autonomous entity embedded in an environment,', 'continuously sensing it and affecting it through actions. An agent is defined by the tuple: Where: = Perception function:', '— maps environment state to an internal percept = Decision function (policy):', "— maps perceived state to an action = Action space — the set of all possible actions = Environment — what the agent operates in The agent function maps the entire percept history to an action: This is important — an agent's decision can depend on everything it has ever seen, not just the", 'current moment. MODULE 1 — Foundations of Agents 🧠 Conc